In [ ]:
# Cell 0 — Clone repo (Colab only)
import os, subprocess
if 'google.colab' in str(get_ipython()):
    REPO_DIR = '/content/trading-strategies'
    if not os.path.exists(os.path.join(REPO_DIR, 'lib', 'data_manager.py')):
        if os.path.exists(REPO_DIR):
            import shutil
            shutil.rmtree(REPO_DIR)
        result = subprocess.run(['git', 'clone', 'https://github.com/r-giov/trading-strategies.git', REPO_DIR], capture_output=True, text=True)
        if result.returncode != 0:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                token = None
            if not token:
                raise RuntimeError("Git clone failed. Add GITHUB_TOKEN as Colab secret.")
            subprocess.run(['git', 'clone', f'https://{token}@github.com/r-giov/trading-strategies.git', REPO_DIR], capture_output=True, text=True)
    assert os.path.isfile(os.path.join(REPO_DIR, 'lib', 'data_manager.py')), "Clone failed"
    os.chdir(REPO_DIR)
    print(f"Repo ready at {REPO_DIR}")
else:
    print("Not in Colab")

In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy
# !pip install matplotlib

In [ ]:
import sys, os

_repo = '/content/trading-strategies'
if os.path.isdir(os.path.join(_repo, 'lib')):
    if _repo not in sys.path:
        sys.path.insert(0, _repo)
else:
    _parent = os.path.join(os.getcwd(), '..')
    if os.path.isdir(os.path.join(_parent, 'lib')) and _parent not in sys.path:
        sys.path.insert(0, os.path.abspath(_parent))

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 140)

print(f"\nPackages: vectorbt={vbt.__version__}, pandas={pd.__version__}, numpy={np.__version__}")
print("All imports loaded")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Momentum Breakout
# ═══════════════════════════════════════════════════════════════

TICKERS = ['GC=F', 'QQQ', 'SPY', 'BTC-USD', 'CL=F', 'SI=F', 'EURUSD=X']
PRIMARY_TICKER = 'GC=F'
START_DATE = '2010-01-01'

print(f"Downloading daily data for {len(TICKERS)} tickers from {START_DATE}...\n")

all_data = {}
for ticker in TICKERS:
    df = yf.download(ticker, start=START_DATE, progress=False)
    if df is not None and not df.empty:
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]
        df = df.dropna()
        all_data[ticker] = df
        print(f"  {ticker}: {len(df)} bars ({df.index[0].date()} to {df.index[-1].date()})")

stock_data = all_data[PRIMARY_TICKER]
TICKER = PRIMARY_TICKER
print(f"\nPrimary: {TICKER} — {len(stock_data)} daily bars")
stock_data.tail(5)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INDICATOR OVERVIEW — Momentum + SMA + ATR + ADX
# ═══════════════════════════════════════════════════════════════

close = stock_data['Close'].values.astype(float)
high = stock_data['High'].values.astype(float)
low = stock_data['Low'].values.astype(float)
idx = stock_data.index

indicators_df = pd.DataFrame({
    'Close': close,
    'MOM_10': talib.MOM(close, timeperiod=10),
    'MOM_14': talib.MOM(close, timeperiod=14),
    'MOM_21': talib.MOM(close, timeperiod=21),
    'SMA_20': talib.SMA(close, timeperiod=20),
    'SMA_50': talib.SMA(close, timeperiod=50),
    'SMA_100': talib.SMA(close, timeperiod=100),
    'SMA_200': talib.SMA(close, timeperiod=200),
    'ATR_14': talib.ATR(high, low, close, timeperiod=14),
    'ADX_14': talib.ADX(high, low, close, timeperiod=14),
}, index=idx)

print(f"Indicators computed for {TICKER}")
print(f"\nMomentum(10) zero-crossings: {((indicators_df['MOM_10'].shift(1) <= 0) & (indicators_df['MOM_10'] > 0)).sum()} bullish")
print(f"Average ATR(14): ${indicators_df['ATR_14'].mean():.2f}")
print(f"Average ADX(14): {indicators_df['ADX_14'].mean():.1f}")
indicators_df.tail(10)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PREPARE PRICE SERIES — IS/OOS SPLIT (60/40)
# ═══════════════════════════════════════════════════════════════

TRAIN_RATIO = 0.60
INIT_CASH   = 100_000
FEES        = 0.0005
SLIPPAGE    = 0.0005

close_series = stock_data['Close'].astype(float)
close_series.name = 'price'
high_series = stock_data['High'].astype(float)
low_series = stock_data['Low'].astype(float)

split_idx = int(len(close_series) * TRAIN_RATIO)
train_close = close_series.iloc[:split_idx]
val_close   = close_series.iloc[split_idx:]

print(f"Total bars: {len(close_series)}")
print(f"Train/Val split at index {split_idx} ({TRAIN_RATIO:.0%}/{1-TRAIN_RATIO:.0%})")
print(f"  Train: {len(train_close)} bars | {train_close.index[0].date()} -> {train_close.index[-1].date()}")
print(f"  Val:   {len(val_close)} bars  | {val_close.index[0].date()} -> {val_close.index[-1].date()}")

## Momentum Breakout — Strategy Description

**Discovered by:** Strategy Discovery Engine (autonomous search across 10,000+ variants)

**Concept:** Momentum measures the rate of price change. When momentum crosses from negative to positive, the asset is accelerating upward. Combined with a trend filter (price > SMA) and optional ADX filter (trending market), this captures the start of new uptrends.

**Entry (Long only):**
- Momentum(period) crosses above 0 (acceleration turns positive)
- Price must be above SMA trend filter (confirms uptrend)
- Optional: ADX must be above threshold (confirms trending, not ranging)

**Exit:**
- Momentum crosses below 0 (acceleration turns negative)
- OR Stop Loss hit (ATR-based)
- OR Take Profit hit (ATR-based)

**Parameters to Optimize:**

| Parameter | Description | Range |
|---|---|---|
| `mom_period` | Momentum lookback | 5, 10, 14, 21 |
| `trend_period` | SMA trend filter | 20, 30, 50, 100, 200 |
| `adx_threshold` | Min ADX for entry (0=off) | 0, 20, 25 |
| `sl_atr_mult` | Stop loss ATR multiplier | None, 1.5, 2.0, 2.5 |
| `tp_atr_mult` | Take profit ATR multiplier | None, 2.0, 3.0, 4.0 |

**Discovery Engine Results (benchmark to beat):**
- GC=F OOS: Sharpe 1.609, Return +47.8%, MaxDD -7.0%, PF 2.68
- QQQ OOS: Sharpe 1.506, Return +44.8%, MaxDD -11.4%, PF 2.43
- SPY OOS: Sharpe 1.108, Return +29.0%, MaxDD -9.8%, PF 1.97

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SIGNAL ENGINE — Momentum Breakout
# ═══════════════════════════════════════════════════════════════

def generate_momentum_signals(close_s, high_s, low_s, mom_period=10,
                               trend_period=50, adx_threshold=0):
    """
    Momentum Breakout with trend + ADX filter.
    Entry: MOM crosses above 0, price > SMA, optional ADX > threshold
    Exit: MOM crosses below 0
    All signals shifted 1 bar for execution delay.
    """
    c = close_s.values.astype(float)
    h = high_s.values.astype(float)
    l = low_s.values.astype(float)
    idx = close_s.index

    mom = pd.Series(talib.MOM(c, timeperiod=mom_period), index=idx)
    trend = pd.Series(talib.SMA(c, timeperiod=trend_period), index=idx)
    price = pd.Series(c, index=idx)

    e_raw = (mom.shift(1) <= 0) & (mom > 0) & (price > trend)

    if adx_threshold > 0:
        adx = pd.Series(talib.ADX(h, l, c, timeperiod=14), index=idx)
        e_raw = e_raw & (adx > adx_threshold)

    x_raw = (mom.shift(1) >= 0) & (mom < 0)

    entries = e_raw.shift(1).fillna(False).astype(bool)
    exits = x_raw.shift(1).fillna(False).astype(bool)
    return entries, exits


def compute_atr_stops(close_s, high_s, low_s, sl_mult=None, tp_mult=None, atr_period=14):
    """Compute ATR-based stops as fraction of price."""
    c = close_s.values.astype(float)
    h = high_s.values.astype(float)
    l = low_s.values.astype(float)
    atr = pd.Series(talib.ATR(h, l, c, timeperiod=atr_period), index=close_s.index)
    sl_stop = (atr * sl_mult / close_s).fillna(0.02) if sl_mult else None
    tp_stop = (atr * tp_mult / close_s).fillna(0.04) if tp_mult else None
    return sl_stop, tp_stop


# Parameter ranges
mom_range   = [5, 10, 14, 21]
trend_range = [20, 30, 50, 100, 200]
adx_range   = [0, 20, 25]
sl_range    = [None, 1.5, 2.0, 2.5]
tp_range    = [None, 2.0, 3.0, 4.0]

all_combos = list(product(mom_range, trend_range, adx_range, sl_range, tp_range))
total_combos = len(all_combos)

print(f"Signal engine defined")
print(f"\nParameter Ranges:")
print(f"  mom_period:     {mom_range}")
print(f"  trend_period:   {trend_range}")
print(f"  adx_threshold:  {adx_range}")
print(f"  sl_atr_mult:    {sl_range}")
print(f"  tp_atr_mult:    {tp_range}")
print(f"\nTotal combinations: {total_combos:,}")

e_test, x_test = generate_momentum_signals(close_series, high_series, low_series)
print(f"\nSanity check ({TICKER}, defaults): Entries={e_test.sum()}, Exits={x_test.sum()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INITIALIZE RESULTS COLLECTION
# ═══════════════════════════════════════════════════════════════

grid_results = []

METRIC_COLS = [
    'mom_period', 'trend_period', 'adx_threshold', 'sl_atr_mult', 'tp_atr_mult',
    'sharpe_ratio', 'sortino_ratio', 'total_return', 'ann_return',
    'max_drawdown', 'volatility', 'calmar_ratio',
    'total_trades', 'trades_per_year', 'win_rate',
    'profit_factor', 'expectancy', 'avg_win', 'avg_loss',
    'largest_win', 'largest_loss', 'payoff_ratio',
]

print(f"Results collection initialized — tracking {len(METRIC_COLS)} metrics per combo")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GRID SEARCH — Training Data Only (IS)
# ═══════════════════════════════════════════════════════════════

print(f"Running grid search on {TICKER} training data ({total_combos:,} combos)...\n")

train_high = high_series.iloc[:split_idx]
train_low  = low_series.iloc[:split_idx]

for combo_idx, (mom_p, trend_p, adx_t, sl_m, tp_m) in enumerate(all_combos):
    if (combo_idx + 1) % 200 == 0:
        print(f"  Combo {combo_idx+1:,}/{total_combos:,}... ({len(grid_results)} valid)")

    try:
        ent, ext = generate_momentum_signals(train_close, train_high, train_low,
                                              mom_period=mom_p, trend_period=trend_p,
                                              adx_threshold=adx_t)
        if ent.sum() < 5:
            continue

        sl_stop, tp_stop = compute_atr_stops(train_close, train_high, train_low,
                                              sl_mult=sl_m, tp_mult=tp_m)
        kw = dict(init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
        if sl_stop is not None: kw['sl_stop'] = sl_stop
        if tp_stop is not None: kw['tp_stop'] = tp_stop

        pf = vbt.Portfolio.from_signals(close=train_close, entries=ent, exits=ext, **kw)
        trades_obj = pf.trades
        n_trades = trades_obj.count()
        if n_trades < 5: continue

        tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
        pos = tr[tr > 0]; neg = tr[tr < 0]
        years = max(len(train_close) / 252, 0.01)

        def safe(fn, default=np.nan):
            try: return float(fn())
            except: return default

        row = {
            'mom_period': mom_p, 'trend_period': trend_p, 'adx_threshold': adx_t,
            'sl_atr_mult': sl_m if sl_m else 0, 'tp_atr_mult': tp_m if tp_m else 0,
            'sharpe_ratio': safe(lambda: pf.sharpe_ratio(freq='D')),
            'sortino_ratio': safe(lambda: pf.sortino_ratio(freq='D')),
            'total_return': safe(pf.total_return),
            'ann_return': safe(lambda: pf.annualized_return(freq='D')),
            'max_drawdown': safe(pf.max_drawdown),
            'volatility': safe(lambda: pf.annualized_volatility(freq='D')),
            'calmar_ratio': np.nan,
            'total_trades': int(n_trades),
            'trades_per_year': n_trades / years,
            'win_rate': float(len(pos)/len(tr)*100) if len(tr) > 0 else np.nan,
            'profit_factor': float(pos.sum()/abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else np.nan,
            'expectancy': float(tr.mean()) if len(tr) > 0 else np.nan,
            'avg_win': float(pos.mean()) if len(pos) > 0 else np.nan,
            'avg_loss': float(neg.mean()) if len(neg) > 0 else np.nan,
            'largest_win': float(pos.max()) if len(pos) > 0 else np.nan,
            'largest_loss': float(neg.min()) if len(neg) > 0 else np.nan,
            'payoff_ratio': float(abs(pos.mean()/neg.mean())) if len(pos) > 0 and len(neg) > 0 else np.nan,
        }

        ann_r, max_dd = row['ann_return'], row['max_drawdown']
        if not np.isnan(ann_r) and not np.isnan(max_dd) and abs(max_dd) > 1e-9:
            row['calmar_ratio'] = ann_r / abs(max_dd)
        grid_results.append(row)
    except: pass

results_df = pd.DataFrame(grid_results)
results_df = results_df.sort_values('sharpe_ratio', ascending=False).reset_index(drop=True)

print(f"\nGrid search complete: {len(results_df):,} valid combos out of {total_combos:,}")
print(f"\nTop 10 by Sharpe Ratio:")
print(f"{'Rank':<5} {'MOM':>5} {'Trend':>6} {'ADX':>5} {'SL':>5} {'TP':>5} {'Sharpe':>8} {'Return':>9} {'MaxDD':>8} {'Trades':>7} {'WR':>7} {'PF':>6}")
print("-" * 80)
for i, row in results_df.head(10).iterrows():
    sl_str = f"{row['sl_atr_mult']:.1f}" if row['sl_atr_mult'] > 0 else "OFF"
    tp_str = f"{row['tp_atr_mult']:.1f}" if row['tp_atr_mult'] > 0 else "OFF"
    print(f"{i+1:<5} {int(row['mom_period']):>5} {int(row['trend_period']):>6} {int(row['adx_threshold']):>5} "
          f"{sl_str:>5} {tp_str:>5} {row['sharpe_ratio']:>8.3f} {row['total_return']:>9.2%} "
          f"{row['max_drawdown']:>8.2%} {int(row['total_trades']):>7} {row['win_rate']:>6.1f}% {row['profit_factor']:>6.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TOP 5 OOS VALIDATION
# ═══════════════════════════════════════════════════════════════

top5 = results_df.head(5).copy()
val_high = high_series.iloc[split_idx:]
val_low  = low_series.iloc[split_idx:]

oos_results = []
is_portfolios = []
oos_portfolios = []

print(f"Validating top 5 IS combos on OOS data...\n")
print(f"{'Rank':<5} {'Params':<35} {'IS SR':>8} {'OOS SR':>8} {'IS Ret':>9} {'OOS Ret':>9} {'IS Tr':>6} {'OOS Tr':>7}")
print("-" * 90)

for rank, (_, row) in enumerate(top5.iterrows()):
    mp = int(row['mom_period']); tp = int(row['trend_period']); adx = int(row['adx_threshold'])
    sl_m = row['sl_atr_mult'] if row['sl_atr_mult'] > 0 else None
    tp_m = row['tp_atr_mult'] if row['tp_atr_mult'] > 0 else None
    sl_str = f"SL={sl_m}" if sl_m else "noSL"
    tp_str = f"TP={tp_m}" if tp_m else "noTP"
    param_str = f"MOM({mp}) T={tp} ADX={adx} {sl_str} {tp_str}"

    # IS
    train_high = high_series.iloc[:split_idx]; train_low = low_series.iloc[:split_idx]
    ent_is, ext_is = generate_momentum_signals(train_close, train_high, train_low,
                                                mom_period=mp, trend_period=tp, adx_threshold=adx)
    sl_is, tp_is = compute_atr_stops(train_close, train_high, train_low, sl_mult=sl_m, tp_mult=tp_m)
    kw_is = dict(init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
    if sl_is is not None: kw_is['sl_stop'] = sl_is
    if tp_is is not None: kw_is['tp_stop'] = tp_is
    pf_is = vbt.Portfolio.from_signals(close=train_close, entries=ent_is, exits=ext_is, **kw_is)
    is_portfolios.append(pf_is)

    # OOS
    ent_oos, ext_oos = generate_momentum_signals(val_close, val_high, val_low,
                                                  mom_period=mp, trend_period=tp, adx_threshold=adx)
    sl_oos, tp_oos = compute_atr_stops(val_close, val_high, val_low, sl_mult=sl_m, tp_mult=tp_m)
    kw_oos = dict(init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
    if sl_oos is not None: kw_oos['sl_stop'] = sl_oos
    if tp_oos is not None: kw_oos['tp_stop'] = tp_oos
    pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=ent_oos, exits=ext_oos, **kw_oos)
    oos_portfolios.append(pf_oos)

    def safe(fn, default=np.nan):
        try: return float(fn())
        except: return default

    is_sr = safe(lambda: pf_is.sharpe_ratio(freq='D'))
    oos_sr = safe(lambda: pf_oos.sharpe_ratio(freq='D'))
    is_ret = safe(pf_is.total_return); oos_ret = safe(pf_oos.total_return)
    is_trades = pf_is.trades.count(); oos_trades = pf_oos.trades.count()

    oos_results.append({
        'rank': rank+1, 'params': param_str,
        'mom_period': mp, 'trend_period': tp, 'adx_threshold': adx,
        'sl_atr_mult': sl_m, 'tp_atr_mult': tp_m,
        'is_sharpe': is_sr, 'oos_sharpe': oos_sr,
        'is_return': is_ret, 'oos_return': oos_ret,
        'is_trades': is_trades, 'oos_trades': oos_trades,
    })
    print(f"{rank+1:<5} {param_str:<35} {is_sr:>8.3f} {oos_sr:>8.3f} {is_ret:>9.2%} {oos_ret:>9.2%} {is_trades:>6} {oos_trades:>7}")

oos_df = pd.DataFrame(oos_results)
best_idx = oos_df['oos_sharpe'].idxmax()
BEST = oos_df.loc[best_idx]
print(f"\nBest OOS combo: MOM({int(BEST['mom_period'])}) Trend={int(BEST['trend_period'])} "
      f"ADX={int(BEST['adx_threshold'])} SL={BEST['sl_atr_mult']} TP={BEST['tp_atr_mult']}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'{TICKER} Momentum Breakout — Top 5 Equity Curves', fontsize=13, fontweight='bold')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i in range(len(is_portfolios)):
    r = oos_results[i]
    is_portfolios[i].value().plot(ax=axes[0], color=colors[i], linewidth=1.5, label=f"#{r['rank']} SR={r['is_sharpe']:.2f}")
    oos_portfolios[i].value().plot(ax=axes[1], color=colors[i], linewidth=1.5, label=f"#{r['rank']} SR={r['oos_sharpe']:.2f}")
for ax, t in zip(axes, ['In-Sample', 'Out-of-Sample']):
    ax.set_title(t); ax.set_ylabel('Portfolio Value ($)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Sensitivity Analysis

Sweep each parameter individually while holding others at best values. Flat bars = LOW sensitivity (robust). Large swings = HIGH sensitivity (fragile).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS
# ═══════════════════════════════════════════════════════════════

best_params = {
    'mom_period': int(BEST['mom_period']), 'trend_period': int(BEST['trend_period']),
    'adx_threshold': int(BEST['adx_threshold']),
    'sl_atr_mult': BEST['sl_atr_mult'], 'tp_atr_mult': BEST['tp_atr_mult'],
}

def run_sensitivity(close_s, high_s, low_s, param_name, param_values, base):
    sharpes = []
    for val in param_values:
        p = base.copy(); p[param_name] = val
        try:
            sig_p = {k: p[k] for k in ['mom_period', 'trend_period', 'adx_threshold']}
            ent, ext = generate_momentum_signals(close_s, high_s, low_s, **sig_p)
            sl_m = p['sl_atr_mult']; tp_m = p['tp_atr_mult']
            sl, tp = compute_atr_stops(close_s, high_s, low_s, sl_mult=sl_m, tp_mult=tp_m)
            kw = dict(init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
            if sl is not None: kw['sl_stop'] = sl
            if tp is not None: kw['tp_stop'] = tp
            pf = vbt.Portfolio.from_signals(close=close_s, entries=ent, exits=ext, **kw)
            sr = float(pf.sharpe_ratio(freq='D'))
            sharpes.append(sr if not np.isnan(sr) else 0.0)
        except: sharpes.append(0.0)
    return sharpes

train_high = high_series.iloc[:split_idx]; train_low = low_series.iloc[:split_idx]

param_configs = [
    ('mom_period', mom_range, int(BEST['mom_period'])),
    ('trend_period', trend_range, int(BEST['trend_period'])),
    ('adx_threshold', adx_range, int(BEST['adx_threshold'])),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle(f'{TICKER} Momentum Breakout — Sensitivity', fontsize=14, fontweight='bold')
sensitivity_summary = []

for col, (pname, pvals, base_val) in enumerate(param_configs):
    is_sharpes = run_sensitivity(train_close, train_high, train_low, pname, pvals, best_params)
    oos_sharpes = run_sensitivity(val_close, val_high, val_low, pname, pvals, best_params)
    base_is = is_sharpes[pvals.index(base_val)] if base_val in pvals else np.mean(is_sharpes)
    base_oos = oos_sharpes[pvals.index(base_val)] if base_val in pvals else np.mean(oos_sharpes)

    for row_idx, (sharpes, prefix, base_sr) in enumerate([(is_sharpes, 'IS', base_is), (oos_sharpes, 'OOS', base_oos)]):
        ax = axes[row_idx][col]; x = np.arange(len(pvals))
        bar_colors = []
        for sr in sharpes:
            pct = (sr - base_sr) / abs(base_sr) * 100 if base_sr != 0 else 0
            if pct > 10: bar_colors.append('darkgreen')
            elif pct > 0: bar_colors.append('lightgreen')
            elif pct > -10: bar_colors.append('orange')
            else: bar_colors.append('red')
        ax.bar(x, sharpes, color=bar_colors, edgecolor='white', alpha=0.85)
        if base_val in pvals:
            ax.axvline(pvals.index(base_val), color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Base: {base_val}')
            ax.legend(fontsize=7)
        ax.set_xticks(x); ax.set_xticklabels([str(v) for v in pvals], fontsize=8)
        ax.set_xlabel(pname); ax.set_ylabel('Sharpe')
        ax.set_title(f'{prefix} — {pname}', fontsize=10, fontweight='bold'); ax.grid(alpha=0.3, axis='y')

    is_r = max(is_sharpes) - min(is_sharpes); oos_r = max(oos_sharpes) - min(oos_sharpes)
    avg = np.mean(is_sharpes + oos_sharpes)
    sens = (is_r + oos_r) / 2 / max(abs(avg), 0.01)
    sensitivity_summary.append((pname, is_r, oos_r, sens, "LOW" if sens < 0.5 else "HIGH"))

plt.tight_layout(); plt.show()
print(f"\nSENSITIVITY SUMMARY:")
print(f"{'Parameter':<20} {'IS Range':>10} {'OOS Range':>10} {'Sensitivity':>12} {'Flag':>8}")
print("-" * 65)
for p, ir, or_, s, f in sensitivity_summary:
    print(f"{p:<20} {ir:>10.3f} {or_:>10.3f} {s:>12.3f} {f:>8}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MULTI-ASSET OOS VALIDATION
# ═══════════════════════════════════════════════════════════════

sig_bp = {k: best_params[k] for k in ['mom_period', 'trend_period', 'adx_threshold']}

print(f"Multi-Asset OOS — Momentum Breakout best params")
print(f"{'='*100}")
print(f"{'Ticker':<10} {'IS SR':>8} {'OOS SR':>8} {'IS Tr':>6} {'OOS Tr':>7} {'OOS Ret':>9} {'OOS DD':>8} {'OOS WR':>8} {'OOS PF':>8}")
print(f"{'-'*100}")

multi_results = []
multi_portfolios = {}

for ticker in TICKERS:
    if ticker not in all_data: continue
    t_df = all_data[ticker]
    t_c = t_df['Close'].astype(float); t_c.name = 'price'
    t_h = t_df['High'].astype(float); t_l = t_df['Low'].astype(float)
    t_split = int(len(t_c) * TRAIN_RATIO)

    def safe(fn, default=np.nan):
        try: return float(fn())
        except: return default

    r = {'ticker': ticker}
    for label, slc in [('is', slice(0, t_split)), ('oos', slice(t_split, None))]:
        try:
            c, h, l = t_c.iloc[slc], t_h.iloc[slc], t_l.iloc[slc]
            ent, ext = generate_momentum_signals(c, h, l, **sig_bp)
            sl, tp = compute_atr_stops(c, h, l, sl_mult=best_params['sl_atr_mult'], tp_mult=best_params['tp_atr_mult'])
            kw = dict(init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
            if sl is not None: kw['sl_stop'] = sl
            if tp is not None: kw['tp_stop'] = tp
            pf = vbt.Portfolio.from_signals(close=c, entries=ent, exits=ext, **kw)
            r[f'{label}_sharpe'] = safe(lambda: pf.sharpe_ratio(freq='D'))
            r[f'{label}_trades'] = pf.trades.count()
            if label == 'oos':
                r['oos_return'] = safe(pf.total_return)
                r['oos_maxdd'] = safe(pf.max_drawdown)
                tr = np.asarray(pf.trades.returns.values if hasattr(pf.trades.returns, 'values') else pf.trades.returns).ravel()
                pos = tr[tr > 0]; neg = tr[tr < 0]
                r['oos_wr'] = float(len(pos)/len(tr)*100) if len(tr) > 0 else np.nan
                r['oos_pf'] = float(pos.sum()/abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else np.nan
            multi_portfolios[(ticker, label)] = pf
        except:
            r[f'{label}_sharpe'] = np.nan; r[f'{label}_trades'] = 0

    multi_results.append(r)
    fmt = lambda v, f: f.format(v) if not np.isnan(v) else "N/A"
    print(f"{ticker:<10} {fmt(r.get('is_sharpe',np.nan),'{:.3f}'):>8} {fmt(r.get('oos_sharpe',np.nan),'{:.3f}'):>8} "
          f"{r.get('is_trades',0):>6} {r.get('oos_trades',0):>7} "
          f"{fmt(r.get('oos_return',np.nan),'{:.2%}'):>9} {fmt(r.get('oos_maxdd',np.nan),'{:.2%}'):>8} "
          f"{fmt(r.get('oos_wr',np.nan),'{:.1f}%'):>8} {fmt(r.get('oos_pf',np.nan),'{:.2f}'):>8}")

cols = min(4, len(TICKERS)); rows = max((len(TICKERS) + cols - 1) // cols, 1)
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
fig.suptitle('Momentum Breakout — OOS Equity Curves', fontsize=13, fontweight='bold')
if rows == 1 and cols > 1: axes = axes.reshape(1, -1)
elif rows == 1 and cols == 1: axes = np.array([[axes]])
axes_flat = axes.flatten()
for i, ticker in enumerate(TICKERS):
    ax = axes_flat[i]; pf = multi_portfolios.get((ticker, 'oos'))
    r = [x for x in multi_results if x['ticker'] == ticker]
    if pf and r:
        pf.value().plot(ax=ax, color='steelblue', linewidth=1.5)
        sr = r[0].get('oos_sharpe', np.nan)
        ax.set_title(f'{ticker} SR={sr:.2f}' if not np.isnan(sr) else ticker, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'No trades', ha='center', va='center'); ax.set_title(ticker, fontsize=9)
    ax.grid(alpha=0.3); ax.tick_params(labelsize=7)
for i in range(len(TICKERS), len(axes_flat)): axes_flat[i].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FTMO MONTE CARLO — 10,000 paths x 30 days
# ═══════════════════════════════════════════════════════════════

import json
try:
    with open(os.path.join('config', 'ftmo_rules.json'), 'r') as f:
        ftmo_cfg = json.load(f)
    FTMO_ACCOUNT = ftmo_cfg['account_size']; FTMO_PROFIT_TARGET = ftmo_cfg['profit_target_pct'] / 100
    FTMO_MAX_DAILY_DD = ftmo_cfg['max_daily_loss_pct'] / 100; FTMO_MAX_TOTAL_DD = ftmo_cfg['max_total_loss_pct'] / 100
    FTMO_CHALLENGE_DAYS = ftmo_cfg['challenge_days']; print("FTMO rules loaded")
except:
    FTMO_ACCOUNT = 100_000; FTMO_PROFIT_TARGET = 0.10
    FTMO_MAX_DAILY_DD = 0.05; FTMO_MAX_TOTAL_DD = 0.10; FTMO_CHALLENGE_DAYS = 30
    print("Using default FTMO rules")

N_PATHS = 10_000
pf_best_oos = oos_portfolios[best_idx]
daily_rets = pf_best_oos.returns().values.ravel()
daily_rets = daily_rets[~np.isnan(daily_rets)]

if len(daily_rets) > 10:
    account = FTMO_ACCOUNT; target = account * FTMO_PROFIT_TARGET
    max_daily = account * FTMO_MAX_DAILY_DD; max_total = account * FTMO_MAX_TOTAL_DD
    n_days = FTMO_CHALLENGE_DAYS

    pass_count = fail_daily = fail_total = fail_time = 0
    final_pnls = []; sample_paths = []

    np.random.seed(42)
    for s in range(N_PATHS):
        sampled = np.random.choice(daily_rets, size=n_days, replace=True)
        daily_pnl = account * sampled; cum_pnl = np.cumsum(daily_pnl)
        equity = account + cum_pnl

        if np.min(daily_pnl) <= -max_daily: fail_daily += 1
        elif np.max(np.maximum.accumulate(equity) - equity) >= max_total: fail_total += 1
        elif np.any(cum_pnl >= target): pass_count += 1
        else: fail_time += 1
        final_pnls.append(cum_pnl[-1])
        if s < 150: sample_paths.append(equity)

    pass_rate = pass_count / N_PATHS; final_pnls = np.array(final_pnls)
    print(f"\nFTMO Monte Carlo — {N_PATHS:,} paths x {n_days} days")
    print(f"{'='*55}")
    print(f"{'Pass Rate':<25} {pass_rate:>15.1%}")
    print(f"{'Fail (daily DD)':<25} {fail_daily/N_PATHS:>15.1%}")
    print(f"{'Fail (total DD)':<25} {fail_total/N_PATHS:>15.1%}")
    print(f"{'Fail (time expired)':<25} {fail_time/N_PATHS:>15.1%}")
    print(f"{'Mean Final P&L':<25} ${final_pnls.mean():>14,.0f}")

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'FTMO Monte Carlo — {TICKER} Momentum | Pass Rate: {pass_rate:.1%}', fontsize=13, fontweight='bold')
    for path in sample_paths:
        c = 'green' if path[-1] >= account + target else ('red' if path[-1] <= account - max_total else 'gray')
        axes[0].plot(path, color=c, alpha=0.12, linewidth=0.5)
    axes[0].axhline(account + target, color='green', linestyle='--', linewidth=2, label=f'+${target:,.0f}')
    axes[0].axhline(account - max_total, color='red', linestyle='--', linewidth=2, label=f'-${max_total:,.0f}')
    axes[0].set_xlabel('Day'); axes[0].set_ylabel('Equity ($)'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[1].hist(final_pnls, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(target, color='green', linestyle='--'); axes[1].axvline(-max_total, color='red', linestyle='--')
    axes[1].set_xlabel('Final P&L ($)'); axes[1].set_ylabel('Freq'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("Not enough data for Monte Carlo")

In [ ]:
# ================================================================
# Universal Export Cell -- Professional PDF Tearsheet + Data Files
# ================================================================
# Uses shared lib/UNIVERSAL_EXPORT_CELL_v2.py for consistent output
# across all strategy notebooks.

STRATEGY_NAME = "Momentum_Breakout"
PARAM_COLS = ['mom_period', 'trend_period', 'adx_threshold', 'sl_atr_mult', 'tp_atr_mult']
# Normalize variables for universal export
try: stock_data
except NameError: stock_data = all_data.get(PRIMARY_TICKER, all_data.get(TICKER))

import os
_lib_dir = 'lib' if os.path.isdir('lib') else os.path.join('..', 'lib')
exec(open(os.path.join(_lib_dir, 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
